# Genotype–Phenotype Heterogeneity and Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's query the dataset for its `recordSet` entities and list their `@id` as well as contained fields and columns, always referencing them by their `@id` values.

In [ ]:
# Display all available record sets by @id and their fields/columns.
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs.get('name')})")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - {field['@id']} ({field.get('name')})")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for col in columns:
            print(f"    - {col['@id']} ({col.get('name')})")
    print()

# Show sample records for each record set
for rs in record_sets:
    print(f"Sample records for RecordSet @id: {rs['@id']} ({rs.get('name')})")
    for rec in dataset.records(record_set=rs['@id']):
        print(rec)
        break  # Just a sample

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will load all record sets into dataframes, using `@id` for indexing.

In [ ]:
# List of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Print columns for the first available record set with data
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in RecordSet {first_rs_id}:", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets in this dataset contain records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming distributions, or grouping data by attributes.

Choose a numeric field and a group field from the columns of one record set (by `@id`).

In [ ]:
import numpy as np

# Pick the first dataframe with numeric columns
selected_rs_id = None
numeric_field_id = None
group_field_id = None
for rs_id, df in dataframes.items():
    for col in df.columns:
        # Try to detect numeric fields
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            selected_rs_id = rs_id
            break
    if numeric_field_id:
        # Try to find a group-able field
        for col in df.columns:
            # Exclude numeric if possible
            if col != numeric_field_id:
                if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
                    group_field_id = col
                    break
        break

if numeric_field_id and selected_rs_id:
    # Filtering by threshold
    threshold = np.nanmean(dataframes[selected_rs_id][numeric_field_id])
    filtered_df = dataframes[selected_rs_id][dataframes[selected_rs_id][numeric_field_id] > threshold]
    print(f"Filtered records in '{selected_rs_id}' with '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No suitable numeric and group fields found for EDA in record sets.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We provide an example histogram for a numeric field, and (if available) a boxplot by group field.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and selected_rs_id:
    plt.figure(figsize=(7,4))
    data = dataframes[selected_rs_id][numeric_field_id].dropna()
    plt.hist(data, bins=10, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {selected_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot if group field is available
    if group_field_id and group_field_id in dataframes[selected_rs_id].columns:
        plt.figure(figsize=(8,4))
        dataframes[selected_rs_id].boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id} in {selected_rs_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 clinical dataset with `mlcroissant` from its schema URL.
- Overview identified multiple record sets, fields, and columns (referenced by their `@id`).
- Data for selected record sets was extracted, filtered, normalized, and grouped by key fields for exploratory analysis.
- Visualization highlighted distribution and relationships in numeric attributes.

Further analysis can focus on more domain-specific questions about genotype-phenotype correlations, limitations due to cohort size, or detailed literature curation fields.